In [1]:
import os
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import gensim
from gensim.models import word2vec

from gensim.models import KeyedVectors #  implements word vectors
from gensim.test.utils import datapath, get_tmpfile
from gensim.scripts.glove2word2vec import glove2word2vec

from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import matplotlib.cm as cm

import spacy

from tqdm.auto import tqdm
tqdm.pandas()

import matplotlib.pyplot as plt
import re

/home/nickolasz/Projects/GoIT/DEEP-LEARNING-FOR-COMPUTER-VISION-AND-NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_path = os.path.expanduser("~/Projects/DataSets/GoIT/iPhone")
datasets_path = os.path.expanduser("~/Projects/DataSets/GoIT")
file_path = os.path.join(dataset_path, "iphone.csv")

df = pd.read_csv(file_path)

df.head()

,productAsin,country,date,isVerified,ratingScore,reviewTitle,reviewDescription,reviewUrl,reviewedIn,variant,variantAsin
0,B09G9BL5CP,India,11-08-2024,True,4,No charger,"Every thing is good about iPhones, there's not...",https://www.amazon.in/gp/customer-reviews/R345...,Reviewed in India on 11 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
1,B09G9BL5CP,India,16-08-2024,True,5,iPhone 13 256GB,"It look so fabulous, I am android user switche...",https://www.amazon.in/gp/customer-reviews/R2HJ...,Reviewed in India on 16 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
2,B09G9BL5CP,India,14-05-2024,True,4,Flip camera option nill,I tried to flip camera while recording but no ...,https://www.amazon.in/gp/customer-reviews/R3Y7...,Reviewed in India on 14 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
3,B09G9BL5CP,India,24-06-2024,True,5,Product,100% genuine,https://www.amazon.in/gp/customer-reviews/R1P9...,Reviewed in India on 24 June 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
4,B09G9BL5CP,India,18-05-2024,True,5,Good product,Happy to get the iPhone 13 in Amazon offer,https://www.amazon.in/gp/customer-reviews/R1XI...,Reviewed in India on 18 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3062 entries, 0 to 3061
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   productAsin        3062 non-null   object
 1   country            3062 non-null   object
 2   date               3062 non-null   object
 3   isVerified         3062 non-null   bool  
 4   ratingScore        3062 non-null   int64 
 5   reviewTitle        3062 non-null   object
 6   reviewDescription  2976 non-null   object
 7   reviewUrl          3046 non-null   object
 8   reviewedIn         3062 non-null   object
 9   variant            3062 non-null   object
 10  variantAsin        3062 non-null   object
dtypes: bool(1), int64(1), object(9)
memory usage: 242.3+ KB


In [4]:
df.dropna(inplace=True)

In [5]:
# відбираємо всі рядки з датафрейму, де в стовпці 'ratingScore' не стоїть значення 3.
df = df.loc[df['ratingScore'] != 3]

In [6]:
#Створення стовпця 'sentiment':
#Якщо ratingScore дорівнює 4 або 5, то в стовпці 'sentiment' буде 1 (позитивний відгук), інакше — 0 (негативний).

df.loc[:, 'sentiment'] = [1 if score in [4, 5] else 0 for score in df['ratingScore']]


df = df.drop_duplicates().reset_index(drop=True)

#Видалення дублікованих рядків за стовпцями 'date' і 'reviewDescription':
df = df.drop_duplicates(subset={"date","reviewDescription"})

# Final size 

df.shape

(2189, 12)

In [7]:
df.head()

,productAsin,country,date,isVerified,ratingScore,reviewTitle,reviewDescription,reviewUrl,reviewedIn,variant,variantAsin,sentiment
0,B09G9BL5CP,India,11-08-2024,True,4,No charger,"Every thing is good about iPhones, there's not...",https://www.amazon.in/gp/customer-reviews/R345...,Reviewed in India on 11 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,1
1,B09G9BL5CP,India,16-08-2024,True,5,iPhone 13 256GB,"It look so fabulous, I am android user switche...",https://www.amazon.in/gp/customer-reviews/R2HJ...,Reviewed in India on 16 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,1
2,B09G9BL5CP,India,14-05-2024,True,4,Flip camera option nill,I tried to flip camera while recording but no ...,https://www.amazon.in/gp/customer-reviews/R3Y7...,Reviewed in India on 14 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,1
3,B09G9BL5CP,India,24-06-2024,True,5,Product,100% genuine,https://www.amazon.in/gp/customer-reviews/R1P9...,Reviewed in India on 24 June 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,1
4,B09G9BL5CP,India,18-05-2024,True,5,Good product,Happy to get the iPhone 13 in Amazon offer,https://www.amazon.in/gp/customer-reviews/R1XI...,Reviewed in India on 18 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,1


In [8]:
contractions = { 
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [9]:
stop_words = set(stopwords.words('english')).union({'also', 'would', 'much', 'many'})

negations = {
    'aren',
    "aren't",
    'couldn',
    "couldn't",
    'didn',
    "didn't",
    'doesn',
    "doesn't",
    'don',
    "don't",
    'hadn',
    "hadn't",
    'hasn',
    "hasn't",
    'haven',
    "haven't",
    'isn',
    "isn't",
    'mightn',
    "mightn't",
    'mustn',
    "mustn't",
    'needn',
    "needn't",
    'no',
    'nor',
    'not',
    'shan',
    "shan't",
    'shouldn',
    "shouldn't",
    'wasn',
    "wasn't",
    'weren',
    "weren't",
    'won',
    "won't",
    'wouldn',
    "wouldn't"
}

stop_words = stop_words.difference(negations)

In [10]:
#Завантаження моделі spaCy en_core_web_sm для обробки англійського тексту, але відключає два компоненти:

#parser — компонент для розбір синтаксичної структури речень.
#ner (Named Entity Recognition) — компонент для розпізнавання іменованих сутностей (наприклад, імена, локації, дати тощо).

nlp = spacy.load("en_core_web_sm", disable = ['parser','ner'])

def normalize_text(raw_review):
    
    # Remove html tags
    text = re.sub("<[^>]*>", " ", raw_review) # match <> and everything in between. [^>] - match everything except >
    
    # Remove emails
    text = re.sub("\S*@\S*[\s]+", " ", text) # match non-whitespace characters, @ and a whitespaces in the end
    
    # remove links
    text = re.sub("https?:\/\/.*?[\s]+", " ", text) # match http, s - zero or once, //, 
                                                    # any char 0-unlimited, whitespaces in the end
        
     # Convert to lower case, split into individual words
    text = text.lower().split()
    
    # Replace contractions with their full versions
    text = [contractions.get(word) if word in contractions else word 
            for word in text]
   
    # Re-splitting for the correct stop-words extraction
    text = " ".join(text).split()    
    
    # Remove stop words
    text = [word for word in text if not word in stop_words]

    text = " ".join(text)
    
    # Remove non-letters        
    text = re.sub("[^a-zA-Z' ]", "", text) # match everything except letters and '


    # Stem words. Need to define porter stemmer above
    # text = [stemmer.stem(word) for word in text.split()]

    # Lemmatize words. Need to define lemmatizer above
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if len(token.lemma_) > 1 ])
    
    # Remove excesive whitespaces
    text = re.sub("[\s]+", " ", text)    
    
    # Join the words back into one string separated by space, and return the result.
    return(text)

<>:14: SyntaxWarning: invalid escape sequence '\S'
<>:17: SyntaxWarning: invalid escape sequence '\/'
<>:47: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\S'
<>:17: SyntaxWarning: invalid escape sequence '\/'
<>:47: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1740143/4240050662.py:14: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub("\S*@\S*[\s]+", " ", text) # match non-whitespace characters, @ and a whitespaces in the end
/tmp/ipykernel_1740143/4240050662.py:17: SyntaxWarning: invalid escape sequence '\/'
  text = re.sub("https?:\/\/.*?[\s]+", " ", text) # match http, s - zero or once, //,
/tmp/ipykernel_1740143/4240050662.py:47: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("[\s]+", " ", text)


In [11]:
from tqdm import tqdm
tqdm.pandas()

df['text_normalized'] = df['reviewDescription'].progress_apply(normalize_text)

100%|██████████| 2189/2189 [00:03<00:00, 702.86it/s]


In [12]:
df.shape

(2189, 13)

In [13]:
def build_corpus(data):
    "Creates a list of lists containing words from each sentence"
    corpus = []
    for sentence in data:
        word_list = sentence.split(" ")
        corpus.append(word_list)    
           
    return corpus

In [14]:
corpus = build_corpus(df['text_normalized'])
corpus[0]

['every',
 'thing',
 'good',
 'iphone',
 'nothing',
 'compare',
 'speed',
 'io',
 'disappoint',
 'no',
 'charger',
 'even',
 'though',
 'indian',
 'judiciary',
 'tell',
 'provide',
 'still',
 'not',
 'make',
 'progress',
 'regard',
 'charger',
 'spend',
 'iphone',
 'brand',
 'acessorie',
 'comfort',
 'size',
 'feature',
 'right',
 'not',
 'point',
 'buying',
 'iphonethank',
 'you']

In [16]:
# Створення моделі
model_emb_from_scratch = word2vec.Word2Vec(corpus, vector_size=100, window=5, min_count=20, workers=4)

# Збереження моделі у форматі binary в середовищі Kaggle
model_emb_from_scratch.wv.save_word2vec_format(dataset_path+'/model_emb_from_scratch.bin', binary=True)
#model_emb_from_scratch.save('/kaggle/working/model_emb_from_scratch_full.model')

In [17]:
class WordEmbedding: 

    def __init__(self):
        self.model = {}
        
    def convert(self, source, ipnut_file_path, output_file_path):
        '''
        Converts word embeddings from GloVe format to Word2Vec format
        '''
        if source == 'glove':
            glove2word2vec(ipnut_file_path, output_file_path)
        elif source in ['word2vec', 'fasttext', 'from_scratch']:
            pass
        else:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
        
    def load(self, source, file_path):
        '''
        Loads a specified word embedding model from a file
        '''
        print(datetime.datetime.now(), 'start: loading', source)
        if source in ['glove', 'fasttext']:
            self.model[source] = gensim.models.KeyedVectors.load_word2vec_format(file_path)
        elif source in ['word2vec', 'from_scratch']:
            self.model[source] = gensim.models.KeyedVectors.load_word2vec_format(file_path, binary=True)
        else:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
            
        print(datetime.datetime.now(), 'end: loading', source)
            
        return self
    
    def get_model(self, source):
        '''
        Retrieves the loaded word embedding model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
            
        return self.model[source]
    
    def get_words(self, source, size=None):
        '''
        Retrieves a list of words from the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        if size is None:
            return [w for w in self.get_model(source=source).key_to_index]
        else:
            results = []
            for i, word in enumerate(self.get_model(source=source).key_to_index):
                if i >= size:
                    break
                results.append(word)
            return results
        
        return Exception('Unexpected flow')
    
    def get_dimension(self, source):
        '''
        Retrieves the dimension of word vectors in the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
        
        return self.get_model(source=source).vectors[0].shape[0]
    
    def get_vectors(self, source, words=None):
        '''
        Retrieves vectors for specified words or for all words in the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
        
        if words is None:
            words = self.get_words(source=source)
            
        embedding = np.empty((len(words), self.get_dimension(source=source)), dtype=np.float32)
        for i, word in enumerate(words):
            embedding[i] = self.get_vector(source=source, word=word)
                
        return embedding
            
    def get_vector(self, source, word):
        '''
        Retrieves the vector representation of a single word
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
            
        if source not in self.model:
            raise ValueError('Did not load %s model yet' % source)
        
        try:
            return self.model[source][word]
        except KeyError as e:
            dims = self.model[source][0].shape
            vect = np.empty(dims)
            vect[:] = np.nan
            return vect
            
    def get_synonym(self, source, word, topn=5):
        '''
        Retrieves synonyms for a given word
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
            
        if source not in self.model:
            raise ValueError('Did not load %s model yet' % source)
        
        try:
            return self.model[source].most_similar(positive=word, topn=topn)
        except KeyError as e:
            raise
    
    def get_distance_between_two_words(self, source, word1, word2):
        '''
        Calculates cosine similarity between two words in the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')
            
        if source not in self.model:
            raise ValueError('Did not load %s model yet' % source)
        
        try:
            return self.model[source].similarity(word1, word2)
        except KeyError as e:
            raise

In [ ]:
word2vec_file_path = '/kaggle/input/google-word2vec/GoogleNews-vectors-negative300.bin'

fasttext_file_path = '/kaggle/input/fasttext-crawl-300d-2m/crawl-300d-2M.vec'

from_scratch_file_path = '/kaggle/working/model_emb_from_scratch.bin'

glove_input_file = '/kaggle/input/glove-embeddings/glove.6B.50d.txt'

word2vec_output_file = '/kaggle/working/glove.6B.50d.word2vec'

In [ ]:
# word2vec_file_path = os.path.join(
#     datasets_path,
#     "GoogleNews-vectors-negative300/GoogleNews-vectors-negative300.bin",
# )

# # fasttext_file_path = '../pretrained_models/wiki-news-300d-1M.vec'
# fasttext_file_path = os.path.join(
#     dataset_path,
#     "fasttext/crawl-300d-2M.vec",
# )
# # from_scratch_file_path = '../saved_models/model_emb_from_scratch.bin'

# # adding absolute path for correct gensim work
# # downloaded_glove_file_path = '../pretrained_models' + '/glove.6B.50d.txt'
# downloaded_glove_file_path = os.path.join(
#     datasets_path,
#     "Glove/glove.6B.50d.txt",
# )

# glove_file_path = os.path.join(
#     dataset_path,
#     "glove.6B.50d.vec",
# )

In [ ]:
import gc
#запускає збірку сміття вручну, звільняємо пам'ять, видаляючи об'єкти, що більше не використовуються.
gc.collect()

In [ ]:
word_embedding = WordEmbedding()

In [ ]:
word_embedding.convert(source='glove', ipnut_file_path=glove_input_file, output_file_path=word2vec_output_file)

In [ ]:
import datetime
word_embedding.load(source='glove', file_path=word2vec_output_file)

In [ ]:
word_embedding.load(source='from_scratch', file_path=from_scratch_file_path)

#model_emb_from_scratch = Word2Vec.load('/kaggle/working/model_emb_from_scratch_full.model')

In [ ]:
for source in ['glove', 'from_scratch']: # 'word2vec', 'fasttext',
    print('Source: %s' % (source))
    print(word_embedding.get_vector(source=source, word='open'))
    print(len(word_embedding.get_vector(source=source, word='open')))

In [ ]:
source = 'word2vec'

word_embedding.load(source=source, file_path=word2vec_file_path)

In [ ]:
def tok2vec(tokens, embedding_model, source='glove', method='mean'):
    """
    Перетворює список токенів у векторний простір на основі вибраної моделі.
    
    tokens: список токенів
    embedding_model: об'єкт класу WordEmbedding
    source: джерело ембедінгу ('glove', 'from_scratch' тощо)
    method: метод агрегації ('mean' або 'sum')
    
    Повертає вектор (np.array)
    """
    # Отримуємо вектори для кожного слова
    vectors = [
        embedding_model.get_vector(source, word)
        for word in tokens
        if not np.isnan(embedding_model.get_vector(source, word)).all()
    ]

    if not vectors:
        return np.zeros(embedding_model.get_dimension(source))  # Повертаємо нульовий вектор, якщо немає векторів

    vectors = np.array(vectors)
    return np.mean(vectors, axis=0) if method == 'mean' else np.sum(vectors, axis=0)


In [ ]:
train_idxs = df.sample(frac=0.8, random_state=42).index
test_idxs = [idx for idx in df.index if idx not in train_idxs]

# Генерація векторів для тренувальних даних
#Для використання власної моделі (яка була навчена раніше за допомогою Word2Vec), необхідно змінити на 'from_scratch'
X_train_raw = df.loc[train_idxs, 'text_normalized'].apply(
    word_tokenize).apply(lambda x: tok2vec(x, word_embedding, 'glove', 'mean')).to_list()

# Генерація векторів для тестових даних
X_test_raw = df.loc[test_idxs, 'text_normalized'].apply(
    word_tokenize).apply(lambda x: tok2vec(x, word_embedding, 'glove', 'mean')).to_list()

In [ ]:
# Перевірка кількості ознак у тренувальних та тестових даних
print(f"Розмір векторів для тренувальних даних (до np.stack): {len(X_train_raw[0])}")
print(f"Розмір векторів для тестових даних (до np.stack): {len(X_test_raw[0])}")


In [ ]:
# Приведення до правильного формату
X_train = np.stack(X_train_raw, axis=0)
X_test = np.stack(X_test_raw, axis=0)

# Перевірка форми після перетворення в масив NumPy
print(f"Форма X_train після np.stack: {X_train.shape}")
print(f"Форма X_test після np.stack: {X_test.shape}")

In [ ]:
# Тренувальні та тестові мітки
y_train = df.loc[train_idxs, 'sentiment']
y_test = df.loc[test_idxs, 'sentiment']

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Прогнозування на тестових даних
y_pred = model.predict(X_test)

# Оцінка точності на тестових даних
accuracy = accuracy_score(y_test, y_pred)
print(f"Точність на тестових даних: {accuracy}")


In [ ]:
texts = [
    "I love this product, it works perfectly",
    "This is the worst purchase I've ever made",
    "Not bad, but could be better",
    "Absolutely fantastic! Highly recommend it",
    "Terrible quality, broke after one use"
]


In [ ]:
new_vectors = [
    tok2vec(word_tokenize(text), word_embedding, 'glove', 'mean')
    for text in texts
]

# Перевірка розміру нових векторів
print(f"Розмір векторів для нових текстів: {len(new_vectors[0])}")

if len(new_vectors[0]) != X_train.shape[1]:
      new_vectors = [np.resize(vec, X_train.shape[1]) for vec in new_vectors]


predictions = model.predict(new_vectors)

for text, pred in zip(texts, predictions):
    print(f"Текст: {text}")
    print(f"Передбачений сентимент: {pred}\n")